# ENG-DRB Claude reproduction

This notebook reproduces the **Claude backend** of the ENG-DRB benchmark using the same Hugging Face data-loading and scoring pipeline as the OpenAI backend.

**Important:** unlike the OpenAI path, this notebook sends **direct Claude requests window by window**. It does **not** use an Anthropic batch file workflow.

In [ ]:
%pip install -q -U anthropic datasets tqdm
import os, sys, json
from pathlib import Path
if not Path('src').exists():
    !git clone YOUR_GITHUB_REPO_URL repo
    %cd repo
sys.path.insert(0, str(Path('src').resolve()))

In [ ]:
from eng_drb_benchmark.data import load_eng_drb, export_gold_jsonl
from eng_drb_benchmark.providers.claude import run_claude_requests
from eng_drb_benchmark.postprocess import merge_claude_results, deduplicate_prediction_file
from eng_drb_benchmark.evaluate import evaluate_from_files

RELATION_TYPE = 'non_implicit'   # or 'implicit'
PROMPT_FILE = f'prompts/{RELATION_TYPE}.txt'
OUTPUT_DIR = Path(f'results/claude_{RELATION_TYPE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL = 'claude-3-7-sonnet-20250219'
WINDOW_SIZE = 20
STEP = 10
CLAUDE_MAX_TOKENS = 3000

In [ ]:
dataset_dict = load_eng_drb()
split_name = next(iter(dataset_dict.keys()))
dataset = dataset_dict[split_name]
gold_path = export_gold_jsonl(dataset, OUTPUT_DIR / f'gold_{RELATION_TYPE}.jsonl', relation_type=RELATION_TYPE)
prompt_text = Path(PROMPT_FILE).read_text(encoding='utf-8')
print(gold_path)

In [ ]:
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_API_KEY_HERE'  # replace in Colab before running
raw_path = run_claude_requests(
    records=(json.loads(line) for line in gold_path.read_text(encoding='utf-8').splitlines() if line.strip()),
    output_path=OUTPUT_DIR / f'claude_raw_{RELATION_TYPE}.jsonl',
    prompt_text=prompt_text,
    model=MODEL,
    max_tokens=CLAUDE_MAX_TOKENS,
    window_size=WINDOW_SIZE,
    step=STEP,
)
print(raw_path)

In [ ]:
merged_path = merge_claude_results(raw_path, OUTPUT_DIR / 'pred_merged.jsonl')
dedup_path = deduplicate_prediction_file(merged_path, OUTPUT_DIR / 'pred_dedup.jsonl')
scores = evaluate_from_files(gold_path, dedup_path)
print(json.dumps(scores['partial_agreement']['overall_scores'], indent=2))
print(json.dumps(scores['exact_match']['overall_scores'], indent=2))